# sLTN — Multi-Digit MNIST Addition

This notebook presents a multi-digit MNIST addition task implemented with **sLTN**, using a vector-valued representation for digits.

The notebook follows the structure:

signature → logic → interpretation → dataset → model → formulas → training → evaluation.

We declare:

- sorts `Image` for MNIST images, `Digit` for probability vectors over classes `0..9`, and `Carry` for probability vectors over `{0, 1}`;
- a structural dimension `p` indexing digit position;
- a function `digit`, grounded by a CNN of type `Image → Digit`, returning a full softmax distribution;
- functions `sumDigit` and `carryOut` of type `Digit × Digit × Carry → Digit` and `Digit × Digit × Carry → Carry`, respectively, grounded as exact lookup-table contractions using `torch.einsum`;
- predicates `=d` and `=c`, which return the probability mass a predicted distribution assigns to the target class, implemented as `dot(pred, target)`.

Digits are categorical symbols rather than points on a number line. Accordingly, `digit(x)` is represented as a full probability vector over classes `0..9`, rather than as a scalar. Sums and carries are then computed through an **exact lookup-table contraction**. 
The addition tables used by `sumDigit` and `carryOut` are exact.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

import sltn

from sltn import Signature, Interpretation
from sltn import Logic, kb_describe
import sltn.interpretation.operators as fuzzy_ops

## §2  Configuration

`N` is the number of digits per operand. `STABLE` and `EPS` numerically stabilise the fuzzy operators. `P` and `P_struct` are the $p$-mean exponents for ordinary and structural quantification, respectively.

In [ ]:
N = 16
num_train_samples = 7500  # chosen freely, independent of N
num_test_samples = 1500
STABLE = True
EPS = 1e-12
P = 4
P_struct = 2
batch_size = 16
nr_of_epochs = 10
learning_rate = 1e-3

device = torch.device("cuda:3" if torch.cuda.is_available() else "mps")

## §3  Logic

We define two `Logic` configurations:

- `logic_train`: a differentiable logic used for gradient-based optimisation, based on product conjunction and $p$-mean aggregators.
- `logic_test`: a hard evaluation logic based on min/max operators, used for reporting.

Both configurations call `.with_defaults(stable=STABLE, eps=EPS)`, so operators such as `Or`, `Implies`, and `Equiv` are derived automatically from the specified `And` and `Not` operators. Quantifiers, knowledge-base aggregation (`SatAgg`), and structural aggregators over `p` are then set explicitly using the bracket syntax, e.g. `logic[symbol] = op`.

In [ ]:
logic_train = Logic(
    And=fuzzy_ops.AndProd(stable=STABLE, eps=EPS),
    Not=fuzzy_ops.NotStandard(),
).with_defaults(stable=STABLE, eps=EPS)
logic_train["∃"] = fuzzy_ops.AggregPMean(stable=STABLE, p=P, eps=EPS)
logic_train["∀"] = fuzzy_ops.AggregPMean(stable=STABLE, p=1/P, eps=EPS)
logic_train["SatAgg"] = fuzzy_ops.AggregPMeanError(stable=STABLE, p=P, eps=EPS)
logic_train["∀,p"] = fuzzy_ops.AggregPMean(stable=STABLE, p=1/P_struct, eps=EPS)
logic_train["∃,p"] = fuzzy_ops.AggregPMean(stable=STABLE, p=P_struct, eps=EPS)

logic_test = Logic(
    And=fuzzy_ops.AndMin(),
    Not=fuzzy_ops.NotStandard(),
).with_defaults(stable=STABLE, eps=EPS)
logic_test["∃"] = fuzzy_ops.AggregMax()
logic_test["∀"] = fuzzy_ops.AggregMin()
logic_test["SatAgg"] = fuzzy_ops.AggregPMean(stable=STABLE, p=P, eps=EPS)
logic_test["∀,p"] = fuzzy_ops.AggregMin()
logic_test["∃,p"] = fuzzy_ops.AggregMax()

print(logic_train)
print(logic_test)

## §4  Dataset — MNIST digit-sum task

Each sample contains two $N$-digit operands, `x1` and `x2`, represented as sequences of MNIST digit images, with one digit image per structural position `p`. Ground-truth sums and carries are produced by simulating standard long addition with carry propagation. Position `0` is the least-significant digit.

**Sampling.** `generate_sums(n, dataset, num_samples)` draws `num_samples` examples. Every digit of both operands — `2 * num_samples * n` MNIST images in total — is drawn independently and uniformly at random, with replacement, from the full `dataset` pool.

As a result, `num_samples` is a free parameter chosen by the caller and does not depend on `n`. This differs from schemes that partition a fixed-size pool into non-overlapping chunks of size `n`, where the number of examples would decrease as `n` grows. Sampling with replacement also means that the same MNIST image may appear in multiple examples, typically paired with different partner digits.

**Re-pairing.** Because every call to `generate_sums` redraws all indices independently, calling it again at the start of each epoch (§9) re-pairs images from scratch. This is important because the knowledge base constrains only *pairs* of images through addition equations, never individual images against their MNIST digit labels directly.

In [ ]:
import torchvision

mnist_train = torchvision.datasets.MNIST(
    "../datasets/", train=True, download=True,
    transform=torchvision.transforms.ToTensor())

mnist_test = torchvision.datasets.MNIST(
    "../datasets/", train=False, download=True,
    transform=torchvision.transforms.ToTensor())

def generate_sums(n, dataset, num_samples):
    """Build `num_samples` random n-digit addition examples from `dataset`. Every digit of
    every operand is drawn independently and uniformly at random, *with replacement*, from
    the whole dataset pool, so `num_samples` is chosen freely by the caller and never
    shrinks as `n` grows (unlike partitioning a fixed-size pool into non-overlapping
    n-sized chunks). Calling this again with the same arguments re-pairs everything from
    scratch (see the re-pairing note in §4/§9)."""
    data_shape = dataset.data[0].shape
    idx1 = torch.randint(0, len(dataset.data), (num_samples, n))
    idx2 = torch.randint(0, len(dataset.data), (num_samples, n))
    data1 = dataset.data[idx1].reshape((num_samples, n) + data_shape).to(torch.float)
    data2 = dataset.data[idx2].reshape((num_samples, n) + data_shape).to(torch.float)
    digit1 = dataset.targets[idx1]
    digit2 = dataset.targets[idx2]
    sums = torch.zeros((num_samples,n),dtype=torch.int)
    carries = torch.zeros((num_samples,n),dtype=torch.int)
    sums[:,0] = (digit1[:,0] + digit2[:,0]) % 10
    carries[:,0] = (digit1[:,0] + digit2[:,0] ) // 10
    for i in range(1,n):
        sums[:,i] = (digit1[:,i] + digit2[:,i] + carries[:,i-1]) % 10
        carries[:,i] = (digit1[:,i] + digit2[:,i] + carries[:,i-1]) // 10
    return {"x1":data1, "x2":data2, "d1":digit1, "d2":digit2, "s":sums, "c":carries}

train_dataset = generate_sums(N, mnist_train, num_samples=num_train_samples)
test_dataset = generate_sums(N, mnist_test, num_samples=num_test_samples)

print(train_dataset["x1"].shape)
print(train_dataset["d1"][0], train_dataset["d2"][0], train_dataset["s"][0], train_dataset["c"][0])
print(test_dataset["d1"][0], test_dataset["d2"][0], test_dataset["s"][0], test_dataset["c"][0])

In [ ]:
class DataLoader:
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.dataset = dataset
        self.dataset_len = dataset["d1"].shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.dataset_len / self.batch_size))

    def __iter__(self):
        idxlist = list(range(self.dataset_len))
        if self.shuffle:
            np.random.shuffle(idxlist)
        for start in range(0, self.dataset_len, self.batch_size):
            idx = idxlist[start:start + self.batch_size]
            yield {k:self.dataset[k][idx] for k in self.dataset} | {"idx":idx}

train_dataloader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)


def plot_sample(samples, idx):
    pics1 = samples["x1"][idx]
    pics2 = samples["x2"][idx]
    fig = plt.figure(figsize=(N,2))
    for i in range(N):
        plt.subplot(2,N,i+1)
        plt.imshow(pics1[i].cpu().squeeze().numpy(),cmap='gray')
        plt.axis('off')
        plt.title(f"{samples['d1'][idx][i]}")
    for i in range(N):
        plt.subplot(2,N,N+i+1)
        plt.imshow(pics2[i].cpu().squeeze().numpy(),cmap='gray')
        plt.axis('off')
        plt.title(f"{samples['d2'][idx][i]}")
    plt.tight_layout()
    plt.show()
    print("sums:", [int(i) for i in samples["s"][idx]])
    print("carries:", [int(i) for i in samples["c"][idx]])

for samples in train_dataloader:
    plot_sample(samples,0)
    break

## §5  Signature

The signature defines the symbolic vocabulary used in the addition task.

| Signature construct | `Signature` method | sLTN code |
|---|---|---|
| signature $\Sigma$ | `Signature(name)` | `sig = Signature("Mnist_sum")` |
| sort `Image` | `sig.sort(name)` | `sig.sort("Image")` |
| sort `Digit` | `sig.sort(name)` | `sig.sort("Digit")` |
| sort `Carry` | `sig.sort(name)` | `sig.sort("Carry")` |
| dimension `p` | `sig.dimension(name)` | `sig.dimension("p")` |
| variables $x_1, x_2$ | `sig.variable(name, sort, dims=…)` | `sig.variable("x1", "Image", dims=["p"])` |
| variable $s$ | `sig.variable(name, sort, dims=…)` | `sig.variable("s", "Digit", dims=["p"])` |
| variable `c_true` | `sig.variable(name, sort)` | `sig.variable("c_true", "Carry")` |
| function `digit` | `sig.function(name, in_sorts, out_sort)` | `sig.function("digit", ["Image"], "Digit")` |
| function `sumDigit` | `sig.function(name, in_sorts, out_sort)` | `sig.function("sumDigit", ["Digit", "Digit", "Carry"], "Digit")` |
| function `carryOut` | `sig.function(name, in_sorts, out_sort)` | `sig.function("carryOut", ["Digit", "Digit", "Carry"], "Carry")` |
| predicate `=d` | `sig.predicate(name, in_sorts, infix=True)` | `sig.predicate("=d", ["Digit", "Digit"], infix=True)` |
| predicate `=c` | `sig.predicate(name, in_sorts, infix=True)` | `sig.predicate("=c", ["Carry", "Carry"], infix=True)` |
| constant `zero_carry` | `sig.constant(name, sort)` | `sig.constant("zero_carry", "Carry")` |

The functions `digit`, `sumDigit`, and `carryOut` each operate at a single structural position. The `p` index is applied at the call site, for example `digit(x1[p=1])`, so these functions do not declare `input_dims` or `output_dims`.

In [ ]:
sig = Signature("Mnist_sum")
sig.sort("Image")
sig.sort("Digit")
sig.sort("Carry")
sig.dimension("p")
sig.variable("x1", "Image", dims=["p"])
sig.variable("x2", "Image", dims=["p"])
sig.variable("s",  "Digit", dims=["p"])
sig.variable("c_true", "Carry")   # ground-truth final carry-out for the whole example
sig.function("digit", ["Image"], "Digit")
sig.function("sumDigit", ["Digit", "Digit", "Carry"], "Digit")
sig.function("carryOut", ["Digit", "Digit", "Carry"], "Carry")
sig.predicate("=d", ["Digit", "Digit"], infix=True)
sig.predicate("=c", ["Carry", "Carry"], infix=True)
sig.constant("zero_carry", "Carry")

# digit constants
for i in range(10):
    sig.constant(f"d{i}", "Digit")

sig

## §6  Formulas and Knowledge Base

The carry chain is defined at the language level using `sig.define`. For each digit position, we introduce a named macro `c{p}(x1, x2)`, which is expanded syntactically at parse time.

For the least-significant digit:

$$
c_0(x_1,x_2) :=
\mathrm{carryOut}\bigl(
\mathrm{digit}(x_1[p{=}0]),
\mathrm{digit}(x_2[p{=}0]),
\mathrm{zero\_carry}
\bigr)
$$

For subsequent positions:

$$
c_p(x_1,x_2) :=
\mathrm{carryOut}\bigl(
\mathrm{digit}(x_1[p{=}p]),
\mathrm{digit}(x_2[p{=}p]),
c_{p-1}(x_1,x_2)
\bigr),
\qquad p = 1, \dots, N{-}1.
$$

Because each `c{p}` is a macro, evaluating it amounts to evaluating the corresponding composition of `digit` and `carryOut`. Gradients therefore flow through the expanded expression into the CNN grounding of `digit`.

The knowledge base is fully unrolled over `p`, since `N` is a small fixed constant known in advance:

$$
\phi_{s,0} :=
\forall (x_1, x_2, s):
s[p{=}0] =_d
\mathrm{sumDigit}\bigl(
\mathrm{digit}(x_1[p{=}0]),
\mathrm{digit}(x_2[p{=}0]),
\mathrm{zero\_carry}
\bigr)
$$

$$
\phi_{s,p} :=
\forall (x_1, x_2, s):
s[p{=}p] =_d
\mathrm{sumDigit}\bigl(
\mathrm{digit}(x_1[p{=}p]),
\mathrm{digit}(x_2[p{=}p]),
c_{p-1}(x_1,x_2)
\bigr),
\qquad p = 1, \dots, N{-}1.
$$

$$
\phi_{c,N} :=
\forall (x_1, x_2, c_\mathrm{true}):
c_{N-1}(x_1,x_2) =_c c_\mathrm{true}
$$

The clauses have the following roles:

- $\phi_{s,0}$ — base case: the least-significant sum digit is computed with incoming carry fixed to `zero_carry`.
- $\phi_{s,p}$ — recursive step: the sum at position `p` uses the carry macro from position `p-1`.
- $\phi_{c,N}$ — final-carry supervision: the last carry is tied to the observed `c_true`.

The notebook also adds symmetry-breaking / class-use clauses over the digit constants `d0, …, d9`. These help discourage degenerate solutions in which the neural predicate collapses multiple digit classes to indistinguishable behaviour.

In [ ]:
# for all i,j with i<j create knowledge base
formulas = []
for i in range(10):
    for j in range(i+1, 10):
        formulas.append(f"∀ x1: ∀ p: (digit(x1[p]) =d d{i}) <-> ∀ x1: ∀ p: (digit(x1[p]) =d d{j})")

formulas = [sig.parse(f) for f in formulas]

formulas

In [ ]:
for p in range(N):
    if p == 0:
        body = "carryOut(digit(x1[p])[p=0], digit(x2[p])[p=0], zero_carry)"
    else:
        body = f"carryOut(digit(x1[p])[p={p}], digit(x2[p])[p={p}], c{p-1}(x1,x2))"
    sig.define(f"c{p}", ["x1", "x2"], body)

phi_s0 = "∀ (x1,x2,s) : (s[p=0] =d sumDigit(digit(x1[p])[p=0], digit(x2[p])[p=0], zero_carry))"
phi_sp = [
    f"∀ (x1,x2,s) : (s[p={p}] =d sumDigit(digit(x1[p])[p={p}], digit(x2[p])[p={p}], c{p-1}(x1,x2)))"
    for p in range(1, N)
]
phi_cN = f"∀ (x1,x2,c_true) : (c{N-1}(x1,x2) =c c_true)"

phi_0_anchor = "∀ (x1, x2) : ((digit(x1[p])[p=0] =d d0) -> (sumDigit(digit(x1[p])[p][p=0], digit(x2[p])[p][p=0], zero_carry) =d digit(x2[p])[p][p=0]))"

# phi_d = [f"∃ x1 : ∃ p : (digit(x1[p])[p] =d d{i})" for i in range(10) for p in range(N)]
# parsed_phi_d = [sig.parse(f) for f in phi_d]

kb = sltn.KB([
    sig.parse(phi_s0),
    *[sig.parse(f) for f in phi_sp],
    sig.parse(phi_cN),
    # sig.parse(phi_0_anchor),
    *formulas
    ])

kb_dict = kb_describe(kb)

## §7  Model and Semantics

**Model.** The function `digit` is grounded by a small CNN (`DigitCNN`) consisting of two convolution-and-pooling blocks followed by two linear layers. The output is a softmax over 10 logits, representing the full probability vector over digit classes. This distribution is retained as a vector and is never collapsed to a scalar.

**Arithmetic.** The functions `sumDigit` and `carryOut` are grounded as exact lookup-table contractions using `torch.einsum`. The tables are defined as follows:

- `sum_table[i, j, k, s] = 1` iff `(i + j + k) % 10 == s`;
- `carry_table[i, j, k, c] = 1` iff `(i + j + k) // 10 == c`;

where `i, j ∈ {0, …, 9}` are digit values and `k ∈ {0, 1}` is the incoming carry.

Contracting these tables against probability vectors gives the exact output distribution induced by the input distributions, under the usual independence assumption. This avoids smoothing discontinuous scalar functions such as `mod10` and `div10`.

**Matching.** The predicates `=d` and `=c` are grounded as `dot(pred, target)`, i.e. the probability mass assigned by the predicted distribution to the target one-hot class. This corresponds to the “probability of truth” used by softmax cross-entropy. Unlike an `exp(-|a-b|)` closeness kernel, the resulting gradient with respect to the logits behind the softmax behaves as `pred - target` and remains informative even when the current prediction is uncertain.

In [ ]:
class DigitCNN(torch.nn.Module):
    """CNN grounding `digit`: Image -> full probability vector over classes 0..9."""
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1   = torch.nn.Linear(32 * 7 * 7, 128)
        self.fc2   = torch.nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        logits = self.fc2(x)
        return torch.softmax(logits, dim=-1)


cnn = DigitCNN()

# Exact addition tables: sum_table[i,j,k,s] = 1 iff (i+j+k)%10==s; carry_table analogous.
sum_table = torch.zeros(10, 10, 2, 10)
carry_table = torch.zeros(10, 10, 2, 2)
for i in range(10):
    for j in range(10):
        for k in range(2):
            total = i + j + k
            sum_table[i, j, k, total % 10] = 1.0
            carry_table[i, j, k, total // 10] = 1.0

In [ ]:
interp = Interpretation(sig).to(device=device)
interp["Image"] = sltn.Type("Image", shape=(1, 28, 28), axis_names=["channel", "height", "width"])
interp["Digit"] = sltn.Type("Digit", shape=(10,), axis_names=["digit_class"])
interp["Carry"] = sltn.Type("Carry", shape=(2,), axis_names=["carry_class"])

interp["digit"] = cnn.to(device=device)

_sum_table = sum_table.to(device=device)
_carry_table = carry_table.to(device=device)
interp["sumDigit"] = lambda d1, d2, cin: torch.einsum("...i,...j,...k,ijks->...s", d1, d2, cin, _sum_table)
interp["carryOut"] = lambda d1, d2, cin: torch.einsum("...i,...j,...k,ijkc->...c", d1, d2, cin, _carry_table)

# Probability mass on the true class, clamped to guard against float rounding past [0, 1].
interp["=d"] = lambda a, b: (a * b).sum(dim=-1, keepdim=True).clamp(0.0, 1.0)
interp["=c"] = lambda a, b: (a * b).sum(dim=-1, keepdim=True).clamp(0.0, 1.0)

interp["zero_carry"] = torch.tensor([1.0, 0.0], device=device) # zero carry class

for i in range(10):
    interp[f"d{i}"] = torch.zeros(10, device=device)
    interp[f"d{i}"][i] = 1.0

## §8  Evaluation Metrics

`compute_metrics` reports digit-recognition accuracy by comparing `argmax digit(x)` against the ground-truth MNIST digit label. This metric is computed independently of whether the downstream sum and carry constraints are satisfied.

The digit labels are used only for evaluation in this section; the knowledge-base training objective uses the addition-derived constraints described above.

In [ ]:
from tqdm.notebook import tqdm


def compute_metrics(interp, dataloader):
    acc = 0
    total = 0
    for samples in tqdm(dataloader):
        interp["x1"] = samples["x1"].unsqueeze(-3).to(device=device)
        interp["x2"] = samples["x2"].unsqueeze(-3).to(device=device)
        preds_d1 = torch.argmax(interp("digit(x1)").value, dim=-1).cpu()
        preds_d2 = torch.argmax(interp("digit(x2)").value, dim=-1).cpu()
        acc += (preds_d1.flatten() == samples["d1"].flatten()).sum()
        acc += (preds_d2.flatten() == samples["d2"].flatten()).sum()
        total += preds_d1.numel() + preds_d2.numel()
    return acc / total

## §9  Training

We train only the neural grounding of `digit`, namely the CNN, against the knowledge base using `logic_train`.

Each epoch proceeds as follows:

1. **Re-pair the data.** `generate_sums` is called again with `num_samples=num_train_samples`. Since every digit is sampled independently with replacement (§4), each epoch sees a new random set of images and pairings.

2. **Bind the batch.** The batch tensors are bound to `x1`, `x2`, `s`, and `c_true`. The sum targets `s` are represented as per-position one-hot digit vectors, and `c_true` is the final carry represented as a one-hot carry vector.

3. **Evaluate and optimise the KB.** The satisfaction value is computed through `interp(kb).value`, and optimisation is performed through the knowledge-base backward step.

4. **Evaluate carry macros implicitly.** There is no separate binding step for intermediate carries. The macros `c{p}(x1, x2)` are expanded by `sig.define` (§6) and evaluated automatically as part of the knowledge-base computation.

5. **Report digit accuracy.** At regular intervals, train and test digit-recognition accuracy are reported with `compute_metrics`. The test set is fixed, providing a stable metric across epochs.

In [ ]:
from sltn import kb_backward

optimizer = torch.optim.Adam(interp["digit"].parameters(), lr=1e-4)

for epoch in range(nr_of_epochs):
    train_dataset = generate_sums(N, mnist_train, num_samples=num_train_samples)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    train_loss = 0.0
    interp["digit"].train()
    interp.logic = logic_train
    for samples in tqdm(train_dataloader):
        interp["x1"] = samples["x1"].unsqueeze(-3).to(device=device)
        interp["x2"] = samples["x2"].unsqueeze(-3).to(device=device)
        interp["s"] = F.one_hot(samples["s"].long(), num_classes=10).float().to(device=device)
        interp["c_true"] = F.one_hot(samples["c"][:, -1].long(), num_classes=2).float().to(device=device)

        optimizer.zero_grad()
        # stats = kb_backward(optimizer, interp, kb_dict, aggregator="upgrad")
        stats = kb_backward(optimizer, interp, kb_dict, aggregator="satagg")
        optimizer.step()

    if epoch % 100 == 0:
        losses_str = " | ".join(
            f"{name}={sat:.3f}" for name, sat in stats["sats"].items()
        )
        w_str = " ".join(f"{w:.2f}" for w in (stats["weights"] or []))
        print(f"epoch {epoch:4d} | [{losses_str}] | α=[{w_str}]")

    if epoch % 1 == 0 or epoch == nr_of_epochs - 1:
        with torch.no_grad():
            train_acc = compute_metrics(interp, train_dataloader)
            test_acc = compute_metrics(interp, test_dataloader)
        print("train_acc, test_acc:", train_acc, test_acc)
        losses_str = " | ".join(
            f"{name}={sat:.3f}" for name, sat in stats["sats"].items()
        )
        w_str = " ".join(f"{w:.2f}" for w in (stats["weights"] or []))
        print(f"epoch {epoch:4d} | [{losses_str}] | α=[{w_str}]")

## §10  Test-set digit-recognition report

After training, we evaluate the learned `digit` grounding directly on MNIST digit labels. This evaluation is diagnostic: the digit labels are not used as direct supervision in the knowledge-base objective.

The classification report and confusion matrix show how well the CNN learned digit identity from the addition constraints alone.

In [ ]:
# ── Test-set classification report + confusion matrix for `digit` ─────────
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

interp["digit"].eval()
digit_names = [str(d) for d in range(10)]

test_preds, test_labels = [], []
with torch.no_grad():
    for samples in test_dataloader:
        interp["x1"] = samples["x1"].unsqueeze(-3).to(device=device)
        interp["x2"] = samples["x2"].unsqueeze(-3).to(device=device)
        preds_d1 = torch.argmax(interp("digit(x1)").value, dim=-1).cpu().flatten()
        preds_d2 = torch.argmax(interp("digit(x2)").value, dim=-1).cpu().flatten()

        test_preds.extend(preds_d1.tolist() + preds_d2.tolist())
        test_labels.extend(samples["d1"].flatten().tolist() + samples["d2"].flatten().tolist())

print(classification_report(test_labels, test_preds, target_names=digit_names))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    test_labels, test_preds, display_labels=digit_names, ax=ax, cmap="Blues",
)
ax.set_title("Digit-recognition confusion matrix (test set)")
plt.tight_layout(); plt.show()